# Sentiment: ViSoBERT fine-tuning (Kaggle GPU)
Fine-tune `5CD-AI/Vietnamese-Sentiment-visobert` (already a 3-class sentiment model) on the hand-checked
VnExpress dev comments. Every dev item is scored out of fold: 5-fold `StratifiedKFold(random_state=42)`, the
same folds as `src/classifiers/sentiment.py`. Three seeds per configuration; the dev scores are the mean
probabilities over seeds. Hyperparameters are fixed in advance (not tuned on dev).

Ablation: the comment alone (`text`) vs the pair (comment, the comment it answers) (`ctx`).

Input: a private Kaggle dataset made from `data/tasks/sentiment/` (`dev.csv`, `corpus.csv`).
Settings: GPU T4 x1 or P100, Internet on. Run only once the dev labels are stable.

Output (download from /kaggle/working into `results/sentiment/kaggle/`):
`dev_scores_visobert_ft_text_oof.csv`, `dev_scores_visobert_ft_ctx_oof.csv`, `visobert_ft_runs.json`.
The last section (run after choosing the final model) writes `corpus_scores_visobert_ft_<cfg>.csv`, the
model in safetensors, and `corpus_scores_visobert_pre_<cfg>.csv` for the pretrained model.
Then locally: `python -m src.classifiers.sentiment collect`.

In [ ]:
import glob, json, os, random, re, time
import numpy as np, pandas as pd, torch
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader
from transformers import (AutoConfig, AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          get_linear_schedule_with_warmup)

DATA = os.path.dirname(glob.glob("/kaggle/input/**/dev.csv", recursive=True)[0])
OUT = "/kaggle/working"
MODEL = "5CD-AI/Vietnamese-Sentiment-visobert"
SEEDS = [42, 43, 44]
EPOCHS, LR, BATCH, MAX_LEN = 4, 2e-5, 16, 256
device = "cuda" if torch.cuda.is_available() else "cpu"

CLASSES = ["neg", "neu", "pos"]
SEED = 42
read = lambda f: pd.read_csv(f"{DATA}/{f}", dtype=str, encoding="utf-8-sig", keep_default_na=False)
dev, corpus = read("dev.csv"), read("corpus.csv")
y = dev["label"].map(CLASSES.index).to_numpy()
mention = re.compile(r"@USER_[0-9a-f]+\s*:?", re.IGNORECASE)
clean = lambda t: mention.sub(" @user ", str(t)).strip()
# Same folds as src/classifiers/sentiment.py, so every item is scored by a model that never saw it
folds = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(dev, y))
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
print(len(dev), dev.label.value_counts().to_dict(), len(corpus), gpu)

tok = AutoTokenizer.from_pretrained(MODEL)
id2label = AutoConfig.from_pretrained(MODEL).id2label          # NEG, POS, NEU: not our order
HEAD = [{v.lower(): k for k, v in id2label.items()}[c] for c in CLASSES]  # head index of neg, neu, pos
print(id2label, HEAD)

In [ ]:
def save_scores(ids, P, name):
    df = pd.DataFrame({"id": list(ids), "pred": [CLASSES[k] for k in P.argmax(1)]})
    for k, c in enumerate(CLASSES):
        df[f"p_{c}"] = P[:, k].round(4)
    df.to_csv(f"{OUT}/{name}.csv", index=False)

def encode(df, use_ctx, labels=None):
    items = []
    for i, (t, p) in enumerate(zip(df["text"], df["parent_text"])):
        pair = clean(p) if use_ctx and p else None
        it = dict(tok(clean(t), pair, truncation=True, max_length=MAX_LEN))
        if labels is not None:
            it["labels"] = HEAD[int(labels[i])]
        items.append(it)
    return items

collate = DataCollatorWithPadding(tok)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def train(df, labels, use_ctx, seed):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL).to(device)
    dl = DataLoader(encode(df, use_ctx, labels), batch_size=BATCH, shuffle=True, collate_fn=collate)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = EPOCHS * len(dl)
    sch = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    model.train()
    for _ in range(EPOCHS):
        for b in dl:
            loss = model(**{k: v.to(device) for k, v in b.items()}).loss
            loss.backward(); opt.step(); sch.step(); opt.zero_grad()
    return model.eval()

@torch.no_grad()
def predict(model, df, use_ctx):
    dl = DataLoader(encode(df, use_ctx), batch_size=64, collate_fn=collate)
    P = [torch.softmax(model(**{k: v.to(device) for k, v in b.items()}).logits.float(), -1).cpu().numpy() for b in dl]
    return np.concatenate(P)[:, HEAD]          # columns in CLASSES order

macro = lambda yt, P: float(f1_score(yt, P.argmax(1), labels=[0, 1, 2], average="macro"))

In [ ]:
runs = []
for use_ctx in (False, True):
    name = f"visobert_ft_{'ctx' if use_ctx else 'text'}_oof"
    per_seed, t0 = [], time.time()
    for seed in SEEDS:
        P = np.zeros((len(dev), 3))
        for tr, te in folds:
            model = train(dev.iloc[tr], y[tr], use_ctx, seed)
            P[te] = predict(model, dev.iloc[te], use_ctx)
            del model; torch.cuda.empty_cache()
        runs.append({"config": name, "seed": seed, "macro_f1": macro(y, P)}); print(runs[-1])
        per_seed.append(P)
    save_scores(dev["id"], np.mean(per_seed, 0), f"dev_scores_{name}")
    print(name, "minutes", round((time.time() - t0) / 60, 1))

json.dump({"runs": runs, "model": MODEL, "epochs": EPOCHS, "lr": LR, "batch": BATCH, "max_len": MAX_LEN,
           "seeds": SEEDS, "gpu": gpu, "folds": "StratifiedKFold(5, shuffle=True, random_state=42)"},
          open(f"{OUT}/visobert_ft_runs.json", "w"), indent=2)
pd.DataFrame(runs).groupby("config")["macro_f1"].agg(["mean", "std"])

## Final model: label the whole corpus
Run after the dev labels are final and the configuration is chosen. Trains on all of dev (3 seeds,
probabilities averaged) and scores `corpus.csv`. Then locally:
`python -m src.classifiers.sentiment label-corpus results/sentiment/kaggle/corpus_scores_visobert_ft_<cfg>.csv`.

In [ ]:
FINAL_CTX = False   # set to the chosen configuration
cfg = "ctx" if FINAL_CTX else "text"
t0, per_seed = time.time(), []
for seed in SEEDS:
    model = train(dev, y, FINAL_CTX, seed)
    per_seed.append(predict(model, corpus, FINAL_CTX))
    if seed == SEEDS[0]:
        model.save_pretrained(f"{OUT}/visobert_sentiment_{cfg}", safe_serialization=True)
        tok.save_pretrained(f"{OUT}/visobert_sentiment_{cfg}")
    del model; torch.cuda.empty_cache()
save_scores(corpus["id"], np.mean(per_seed, 0), f"corpus_scores_visobert_ft_{cfg}")
print("minutes", round((time.time() - t0) / 60, 1))

In [ ]:
# Pretrained model (no training) on the whole corpus, for comparison or as the final model
pre = AutoModelForSequenceClassification.from_pretrained(MODEL).to(device).eval()
for use_ctx in (False, True):
    save_scores(corpus["id"], predict(pre, corpus, use_ctx), f"corpus_scores_visobert_pre_{'ctx' if use_ctx else 'text'}")